# Clinical Threshold Extension

**Source script:** `clinical_threshold_apriori_extension.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

import argparse
import re
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

from run_eda_v3 import (
    SEED,
    add_moon_cyclic_features,
    build_feature_blocks,
    classify_columns,
    derive_temporal_columns,
    infer_date_column,
    infer_outcome_column,
    load_dataset,
)

ROOT = Path(__file__).resolve().parent
DEFAULT_INPUT = ROOT / "outputs" / "imputed_dataset_enriched.csv"
DEFAULT_OUTDIR = ROOT / "outputs" / "eda_v3" / "clinical_threshold_extension"
SYNODIC_MONTH_DAYS = 29.530588


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Clinical threshold extension + severity Apriori.")
    parser.add_argument("--input", type=Path, default=DEFAULT_INPUT, help="Input enriched dataset")
    parser.add_argument("--outdir", type=Path, default=DEFAULT_OUTDIR, help="Output directory")
    parser.add_argument("--seed", type=int, default=SEED, help="Random seed")
    parser.add_argument("--corr-threshold", type=float, default=0.85, help="Environmental correlation threshold")
    parser.add_argument("--window-days", type=float, default=2.0, help="Moon near-window in days")
    parser.add_argument("--min-support", type=float, default=0.05, help="Apriori min support")
    parser.add_argument("--min-lift", type=float, default=1.20, help="Apriori min lift")
    return parser.parse_args()


def normalize_name(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())


def sanitize_item_name(text: str) -> str:
    out = re.sub(r"[^a-z0-9]+", "_", str(text).lower()).strip("_")
    return out or "unknown"



CLINICAL_THRESHOLDS: Dict[str, Dict[str, object]] = {
    "hematocrit": {
        "patterns": ["initialhematocrit", "hematocrit"],
        "state_col": "hematocrit_state",
        "type": "four_state",
        "increased_min": 52.4,
        "upper_min": 45.0,
        "phys_min": 37.7,
    },
    "leukocyte": {
        "patterns": ["leukocytecount"],
        "state_col": "leukocyte_state",
        "type": "four_state",
        "increased_min": 17.03,
        "upper_min": 12.28,
        "phys_min": 7.58,
    },
    "neutrophil": {
        "patterns": ["neutrophilcount"],
        "state_col": "neutrophil_state",
        "type": "four_state",
        "increased_min": 10.3,
        "upper_min": 7.35,
        "phys_min": 4.42,
    },
    "lymphocyte": {
        "patterns": ["lymphocytecount"],
        "state_col": "lymphocyte_state",
        "type": "four_state",
        "increased_min": 6.89,
        "upper_min": 5.0,
        "phys_min": 2.92,
    },
    "total_protein": {
        "patterns": ["initialtotalproteinconcentration", "totalprotein"],
        "state_col": "total_protein_state",
        "type": "four_state",
        "increased_min": 79.0,
        "upper_min": 72.0,
        "phys_min": 65.0,
    },
    "albumin": {
        "patterns": ["albuminconcentration", "albumin"],
        "state_col": "albumin_state",
        "type": "four_state",
        "increased_min": 36.0,
        "upper_min": 33.0,
        "phys_min": 28.0,
    },
    "bun": {
        "patterns": ["bunconcentration", "bun"],
        "state_col": "bun_state",
        "type": "four_state",
        "increased_min": 11.72,
        "upper_min": 9.92,
        "phys_min": 8.10,
    },
    "creatinine": {
        "patterns": ["creaconcentration", "crea", "creatinine"],
        "state_col": "creatinine_state",
        "type": "four_state",
        "increased_min": 160.0,
        "upper_min": 131.0,
        "phys_min": 101.0,
    },
    "serum_k": {
        "patterns": ["kvalue", "serumk"],
        "state_col": "serum_k_state",
        "type": "four_state",
        "increased_min": 4.7,
        "upper_min": 4.3,
        "phys_min": 3.9,
    },
    "serum_na": {
        "patterns": ["navalue", "serumna"],
        "state_col": "serum_na_state",
        "type": "four_state",
        "increased_min": 157.0,
        "upper_min": 154.0,
        "phys_min": 151.0,
    },
    "serum_cl": {
        "patterns": ["clvalue", "serumcl"],
        "state_col": "serum_cl_state",
        "type": "four_state",
        "increased_min": 121.0,
        "upper_min": 117.0,
        "phys_min": 112.0,
    },
    "pancreatic_lipase": {
        "patterns": ["pancreaticlipase", "lipase", "fpl"],
        "state_col": "pancreatic_lipase_state",
        "type": "lipase",
    },
    "t4": {
        "patterns": ["t4"],
        "state_col": "t4_state",
        "type": "t4",
    },
}


def find_data_column(df: pd.DataFrame, patterns: Sequence[str]) -> str | None:
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    for c in numeric_candidates:
        n = normalize_name(c)
        if "__missing" in c.lower():
            continue
        if any(p in n for p in patterns):
            return c
    return None


def map_state_four(v: float, increased_min: float, upper_min: float, phys_min: float) -> str:
    if pd.isna(v):
        return "missing"
    if float(v) >= float(increased_min):
        return "increased"
    if float(v) >= float(upper_min):
        return "upper_limit"
    if float(v) >= float(phys_min):
        return "physiological"
    return "lower_limit"


def map_state_lipase(v: float) -> str:
    if pd.isna(v):
        return "missing"
    if float(v) >= 5.4:
        return "likely_pancreatitis"
    if float(v) >= 3.6:
        return "possible_pancreatitis"
    return "normal"


def map_state_t4(v: float) -> str:
    if pd.isna(v):
        return "missing"
    if float(v) >= 61.0:
        return "possible_hyperthyreosis"
    if float(v) >= 30.0:
        return "grey_range"
    if float(v) >= 10.0:
        return "normal"
    return "subnormal"


def derive_clinical_states(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, str]]:
    out = df.copy()
    dist_rows: List[Dict[str, object]] = []
    source_cols: Dict[str, str] = {}

    for lab_key, cfg in CLINICAL_THRESHOLDS.items():
        col = find_data_column(out, cfg["patterns"])
        if col is None:
            continue
        source_cols[lab_key] = col
        s = pd.to_numeric(out[col], errors="coerce")
        state_col = str(cfg["state_col"])

        if cfg["type"] == "four_state":
            out[state_col] = s.map(
                lambda v: map_state_four(
                    v=v,
                    increased_min=float(cfg["increased_min"]),
                    upper_min=float(cfg["upper_min"]),
                    phys_min=float(cfg["phys_min"]),
                )
            )
        elif cfg["type"] == "lipase":
            out[state_col] = s.map(map_state_lipase)
        elif cfg["type"] == "t4":
            out[state_col] = s.map(map_state_t4)
        else:
            continue

        vc = out[state_col].value_counts(dropna=False)
        total = len(out)
        for state, count in vc.items():
            dist_rows.append(
                {
                    "lab_key": lab_key,
                    "source_column": col,
                    "state_column": state_col,
                    "state": str(state),
                    "count": int(count),
                    "pct": float(100.0 * count / total),
                }
            )

    dist_df = (
        pd.DataFrame(dist_rows)
        .sort_values(["lab_key", "state"], ascending=[True, True])
        .reset_index(drop=True)
        if dist_rows
        else pd.DataFrame(columns=["lab_key", "source_column", "state_column", "state", "count", "pct"])
    )
    return out, dist_df, source_cols


def add_severity_flags(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    out = df.copy()
    created: List[str] = []

    mapping = {
        "leukocyte_increased": ("leukocyte_state", "increased"),
        "neutrophil_increased": ("neutrophil_state", "increased"),
        "lymphocyte_increased": ("lymphocyte_state", "increased"),
        "albumin_low": ("albumin_state", "lower_limit"),
        "BUN_increased": ("bun_state", "increased"),
        "creatinine_increased": ("creatinine_state", "increased"),
        "lipase_likely": ("pancreatic_lipase_state", "likely_pancreatitis"),
        "T4_high": ("t4_state", "possible_hyperthyreosis"),
    }
    for flag, (state_col, target_state) in mapping.items():
        if state_col in out.columns:
            out[flag] = out[state_col].astype(str).eq(target_state)
            created.append(flag)

    return out, created


def correlation_filter_like_modeling(df: pd.DataFrame, weather_numeric_cols: Sequence[str], threshold: float) -> List[str]:
    if not weather_numeric_cols:
        return []
    X = df[list(weather_numeric_cols)].copy()
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))
    corr = X.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if (upper[col] > threshold).any()]
    return [c for c in weather_numeric_cols if c not in to_drop]


def circular_distance_deg(angle_deg: np.ndarray, anchor_deg: float) -> np.ndarray:
    d = np.abs(angle_deg - anchor_deg)
    return np.minimum(d, 360.0 - d)


def add_lunar_window_flags(df: pd.DataFrame, window_days: float) -> pd.DataFrame:
    out = df.copy()
    window_deg = float(window_days) * (360.0 / SYNODIC_MONTH_DAYS)

    angle_col = None
    for c in out.columns:
        if normalize_name(c) == "moonphaseangledeg":
            angle_col = c
            break
    if angle_col is None:
        out["moon_near_full"] = False
        out["moon_near_new"] = False
        return out

    angle = pd.to_numeric(out[angle_col], errors="coerce").to_numpy(dtype=float)
    angle = np.mod(angle, 360.0)

    out["moon_near_new"] = pd.Series(circular_distance_deg(angle, 0.0) <= window_deg, index=out.index).fillna(False)
    out["moon_near_full"] = pd.Series(circular_distance_deg(angle, 180.0) <= window_deg, index=out.index).fillna(False)
    out["moon_near_new"] = out["moon_near_new"].astype(bool)
    out["moon_near_full"] = out["moon_near_full"].astype(bool)
    return out


def build_model_aligned_environment(df: pd.DataFrame, corr_threshold: float) -> Tuple[pd.DataFrame, List[str], List[str]]:
    outcome_col = infer_outcome_column(df)
    date_col = infer_date_column(df)
    out, temporal_cols = derive_temporal_columns(df, date_col)
    out, _ = add_moon_cyclic_features(out)
    col_groups = classify_columns(out, outcome_col, temporal_cols)

    kept_weather_numeric = correlation_filter_like_modeling(
        df=out,
        weather_numeric_cols=col_groups["weather_model_numeric"],
        threshold=corr_threshold,
    )
    feature_blocks = build_feature_blocks(col_groups)
    env_numeric_original = set(col_groups["weather_model_numeric"])
    feature_blocks["environmental_only"]["numeric"] = sorted(kept_weather_numeric)
    feature_blocks["combined"]["numeric"] = sorted(
        [c for c in feature_blocks["combined"]["numeric"] if c not in env_numeric_original]
        + list(kept_weather_numeric)
    )

    env_numeric = list(feature_blocks["environmental_only"]["numeric"])
    env_categorical = list(feature_blocks["environmental_only"]["categorical"])
    return out, env_numeric, env_categorical


def build_environmental_items(df: pd.DataFrame, env_numeric: Sequence[str], env_categorical: Sequence[str]) -> pd.DataFrame:
    items: Dict[str, pd.Series] = {}


    numeric_for_bins = [c for c in env_numeric if normalize_name(c) not in {"moonphasesin", "moonphasecos"}]

    for c in numeric_for_bins:
        s = pd.to_numeric(df[c], errors="coerce")
        valid = s.dropna()
        if valid.empty:
            continue
        q25 = float(valid.quantile(0.25))
        q75 = float(valid.quantile(0.75))
        if q25 >= q75:
            continue
        low = (s <= q25).fillna(False)
        high = (s >= q75).fillna(False)
        low_name = f"{sanitize_item_name(c)}_low"
        high_name = f"{sanitize_item_name(c)}_high"
        if bool(low.any()):
            items[low_name] = low.astype(bool)
        if bool(high.any()):
            items[high_name] = high.astype(bool)

    for c in env_categorical:
        s = df[c].astype(str).str.strip()
        s = s.replace({"": "missing", "nan": "missing", "None": "missing", "NaN": "missing"}).fillna("missing")
        for val in sorted(s.unique().tolist()):
            name = f"{sanitize_item_name(c)}__{sanitize_item_name(val)}"
            items[name] = s.eq(val).astype(bool)

    for moon_flag in ["moon_near_full", "moon_near_new"]:
        if moon_flag in df.columns:
            items[moon_flag] = df[moon_flag].fillna(False).astype(bool)

    tx = pd.DataFrame(items, index=df.index).fillna(False).astype(bool)
    return tx


def mine_severity_rules(
    tx_env: pd.DataFrame,
    df: pd.DataFrame,
    severity_flags: Sequence[str],
    min_support: float,
    min_lift: float,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    n_rows = len(tx_env)
    antecedent_cols = set(tx_env.columns.tolist())

    all_rows: List[pd.DataFrame] = []
    baseline_rows: List[Dict[str, object]] = []

    for flag in severity_flags:
        y = df[flag].fillna(False).astype(bool)
        baseline = float(y.mean())
        baseline_rows.append({"severity_marker": flag, "baseline_prevalence": baseline})
        if baseline <= 0:
            continue

        mat = pd.concat([tx_env, y.rename(flag)], axis=1).fillna(False).astype(bool)
        itemsets = apriori(mat, min_support=min_support, use_colnames=True, max_len=4)
        if itemsets.empty:
            continue
        rules = association_rules(itemsets, metric="confidence", min_threshold=baseline)
        if rules.empty:
            continue

        rules = rules[
            rules["antecedents"].apply(lambda s: 1 <= len(s) <= 3 and set(s).issubset(antecedent_cols))
            & rules["consequents"].apply(lambda s: len(s) == 1 and flag in s)
            & (rules["support"] >= min_support)
            & (rules["lift"] >= min_lift)
            & (rules["confidence"] >= baseline)
        ].copy()
        if rules.empty:
            continue

        rules["severity_marker"] = flag
        rules["antecedents"] = rules["antecedents"].apply(lambda s: ", ".join(sorted([str(x) for x in s])))
        rules["support_count"] = (rules["support"] * n_rows).round().astype(int)
        rules["baseline_prevalence"] = baseline
        rules = rules.sort_values(["lift", "confidence"], ascending=[False, False]).head(10)
        keep = ["severity_marker", "antecedents", "support", "support_count", "confidence", "lift", "baseline_prevalence"]
        for optional in ["leverage", "conviction"]:
            if optional in rules.columns:
                keep.append(optional)
        all_rows.append(rules[keep].reset_index(drop=True))

    rules_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame(
        columns=[
            "severity_marker",
            "antecedents",
            "support",
            "support_count",
            "confidence",
            "lift",
            "baseline_prevalence",
            "leverage",
            "conviction",
        ]
    )
    baseline_df = pd.DataFrame(baseline_rows).sort_values("severity_marker").reset_index(drop=True)
    return rules_df, baseline_df


def to_md_table(df: pd.DataFrame, cols: Sequence[str]) -> str:
    if df.empty:
        return "_No rows._"
    use_cols = [c for c in cols if c in df.columns]
    lines = []
    lines.append("| " + " | ".join(use_cols) + " |")
    lines.append("| " + " | ".join(["---"] * len(use_cols)) + " |")
    for _, row in df[use_cols].iterrows():
        vals = []
        for c in use_cols:
            v = row[c]
            if isinstance(v, float):
                vals.append(f"{v:.4f}")
            else:
                vals.append(str(v))
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)


def write_summary(
    out_file: Path,
    baseline_df: pd.DataFrame,
    rules_df: pd.DataFrame,
    dist_df: pd.DataFrame,
    env_numeric: Sequence[str],
    env_categorical: Sequence[str],
) -> None:
    counts = (
        rules_df.groupby("severity_marker", dropna=False)
        .size()
        .rename("n_rules")
        .reset_index()
        .sort_values(["n_rules", "severity_marker"], ascending=[False, True])
        .reset_index(drop=True)
        if not rules_df.empty
        else pd.DataFrame(columns=["severity_marker", "n_rules"])
    )

    lines: List[str] = []
    lines.append("# Severity Rules Summary (Clinical Threshold Extension)")
    lines.append("")
    lines.append("## Setup")
    lines.append(f"- Environmental numeric (kept): {len(env_numeric)}")
    lines.append(f"- Environmental categorical: {len(env_categorical)}")
    lines.append("- Consequents: severity flags only (clinical threshold states).")
    lines.append("")
    lines.append("## Baseline Prevalence Per Severity Marker")
    lines.append(to_md_table(baseline_df, ["severity_marker", "baseline_prevalence"]))
    lines.append("")
    lines.append("## Rule Count Per Severity Marker")
    lines.append(to_md_table(counts, ["severity_marker", "n_rules"]))
    lines.append("")
    lines.append("## Top 3 Strongest Rules Per Marker")
    if rules_df.empty:
        lines.append("- No rules met thresholds.")
    else:
        for marker in sorted(baseline_df["severity_marker"].tolist()):
            sub = rules_df[rules_df["severity_marker"] == marker].sort_values(["lift", "confidence"], ascending=[False, False]).head(3)
            lines.append(f"### {marker}")
            if sub.empty:
                lines.append("- No rules met thresholds.")
            else:
                lines.append(
                    to_md_table(
                        sub,
                        ["antecedents", "support", "confidence", "lift", "baseline_prevalence"],
                    )
                )
    lines.append("")
    lines.append("## Clinical State Coverage")
    lines.append(to_md_table(dist_df, ["lab_key", "state", "count", "pct"]))
    lines.append("")
    lines.append("## Note")
    lines.append("- If a severity marker has zero qualifying rules, thresholds were still applied but no association passed filters.")

    out_file.write_text("\n".join(lines), encoding="utf-8")


def main() -> None:
    args = parse_args()
    np.random.seed(args.seed)
    args.outdir.mkdir(parents=True, exist_ok=True)

    df = load_dataset(args.input)
    drop_weather_reason = [c for c in df.columns if "weather_missing_reason" in c.lower()]
    if drop_weather_reason:
        df = df.drop(columns=drop_weather_reason)

    df, env_numeric, env_categorical = build_model_aligned_environment(df, args.corr_threshold)
    df = add_lunar_window_flags(df, window_days=args.window_days)
    df, dist_df, source_cols = derive_clinical_states(df)
    df, severity_flags = add_severity_flags(df)

    out_dataset = args.outdir / "imputed_dataset_enriched_with_clinical_states.csv"
    out_dist = args.outdir / "clinical_state_distribution.csv"
    out_rules = args.outdir / "severity_rules_clinical_thresholds.csv"
    out_summary = args.outdir / "severity_rules_summary.md"

    df.to_csv(out_dataset, index=False)
    dist_df.to_csv(out_dist, index=False)

    tx_env = build_environmental_items(df, env_numeric, env_categorical)
    rules_df, baseline_df = mine_severity_rules(
        tx_env=tx_env,
        df=df,
        severity_flags=severity_flags,
        min_support=args.min_support,
        min_lift=args.min_lift,
    )
    rules_df.to_csv(out_rules, index=False)
    write_summary(
        out_file=out_summary,
        baseline_df=baseline_df,
        rules_df=rules_df,
        dist_df=dist_df,
        env_numeric=env_numeric,
        env_categorical=env_categorical,
    )


    print("=== Clinical Threshold Extension ===")
    print(f"Output directory: {args.outdir}")
    print(f"Rows: {len(df)}")
    print(f"Clinical state columns created from labs: {len(source_cols)}")
    for lab_key in sorted(source_cols.keys()):
        print(f"  {lab_key}: source={source_cols[lab_key]}")
        sub = dist_df[dist_df["lab_key"] == lab_key].sort_values("state")
        for _, r in sub.iterrows():
            print(f"    - {r['state']}: {int(r['count'])} ({float(r['pct']):.2f}%)")

    print("Severity markers and qualifying rule counts:")
    for marker in severity_flags:
        n_rules = int((rules_df["severity_marker"] == marker).sum()) if not rules_df.empty else 0
        print(f"  {marker}: {n_rules}")

    if rules_df.empty:
        print("Strongest rule: none (no rules met threshold)")
    else:
        top = rules_df.sort_values(["lift", "confidence"], ascending=[False, False]).iloc[0]
        print(
            "Strongest rule overall: "
            f"{top['severity_marker']} <= {top['antecedents']} "
            f"(lift={top['lift']:.3f}, conf={top['confidence']:.3f}, support={top['support']:.3f})"
        )


if __name__ == "__main__":
    main()
